# Train YOLOv8s on the ROD-Dataset

Real-Time Obstacle Detection — 25 classes, 24,326 images.

## Before you run
1. **Settings → Accelerator → `GPU T4 x2`**
2. **Settings → Internet → ON**
3. **Add Data** → search `obstacle-detection-dataset` by `abtinzandi`

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch, ultralytics

print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')

## Locate the dataset and write a fresh `data.yaml`

`/kaggle/input/` is read-only, so we generate a corrected `data.yaml` in `/kaggle/working/`.

In [ ]:
import yaml
from pathlib import Path

# Dataset path
INPUT_ROOT = Path('/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset')

src = next(INPUT_ROOT.rglob('data.yaml'), None)
assert src is not None, f'data.yaml not found under {INPUT_ROOT}'
data_root = src.parent
print('Dataset root:', data_root)

with open(src) as f:
    orig = yaml.safe_load(f)

data_yaml = {
    'path':  str(data_root),
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    orig['nc'],
    'names': orig['names'],
}

out = Path('/kaggle/working/data.yaml')
with open(out, 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print('Wrote', out)
print('Classes:', data_yaml['nc'])
print(data_yaml['names'])

## Train

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data    = '/kaggle/working/data.yaml',
    task    = 'detect',
    epochs  = 100,
    imgsz   = 640,
    batch   = 32,
    device  = 0,
    amp     = True,
    cache   = False,
    workers = 4,
    project = '/kaggle/working/runs',
    name    = 'rod_yolov8s',
    patience= 20,
    cos_lr  = True,
    plots   = True,
    seed    = 42,
)

## Validate on the held-out test split
Reports mAP50, mAP50-95, precision, recall.

In [ ]:
# Reload best.pt — more reliable than using the training model object
best = Path(results.save_dir) / 'weights' / 'best.pt'
eval_model = YOLO(str(best))

metrics = eval_model.val(
    data   = '/kaggle/working/data.yaml',
    task   = 'detect',
    split  = 'test',
    imgsz  = 640,
    device = 0,
    plots  = True,
)
print(metrics.results_dict)

## Show results — training curves, confusion matrix, PR curve

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_image(path, title=''):
    path = Path(path)
    if not path.exists():
        print(f'[skip] {path.name} not found')
        return
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.imshow(mpimg.imread(str(path)))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

run_dir = Path(results.save_dir)
val_dir = Path(metrics.save_dir)

show_image(run_dir / 'results.png',                        'Training curves')
show_image(val_dir / 'confusion_matrix_normalized.png',    'Confusion Matrix (normalised)')
show_image(val_dir / 'confusion_matrix.png',               'Confusion Matrix (raw)')
show_image(val_dir / 'PR_curve.png',                       'Precision-Recall Curve')
show_image(val_dir / 'F1_curve.png',                       'F1-Confidence Curve')

## Predict on a handful of test images and display them

In [ ]:
import random, glob

test_imgs = sorted(glob.glob(str(data_root / 'test/images/*.jpg')))
samples = random.sample(test_imgs, 6)

preds = eval_model.predict(samples, imgsz=640, conf=0.25, save=True,
                           project='/kaggle/working/runs', name='predict')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, p in zip(axes.flat, sorted(glob.glob('/kaggle/working/runs/predict/*.jpg'))[:6]):
    ax.imshow(mpimg.imread(p))
    ax.axis('off')
plt.suptitle('YOLOv8s predictions (conf ≥ 0.25)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Copy best weights to `/kaggle/working/` for download

In [ ]:
import shutil

for src in (run_dir / 'weights' / 'best.pt', run_dir / 'weights' / 'last.pt'):
    if src.exists():
        dst = Path('/kaggle/working') / src.name
        shutil.copy(str(src), str(dst))
        print('Saved', dst)

## (Optional) Export for mobile / ONNX

In [ ]:
# eval_model.export(format='onnx', imgsz=640, opset=12)
# eval_model.export(format='tflite', imgsz=320, int8=True)
# eval_model.export(format='ncnn', imgsz=320)